---
title: "Run a managed workflow"
description: "Run a managed workflow, check its status, result, and logs, find the cause of an error, and cancel a job."
---

{/* cspell:ignore maxwait squeue scancel */}

# Run a managed workflow

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-serverless~=0.37.0
```
</AccordionItem>
</Accordion>

<Admonition type="caution" title="Placeholder content">
- **Class names.** The code examples import `QiskitServerless` as `ManagedWorkflowClient` and `QiskitFunction` as `ManagedWorkflow`. Replace the aliases with the new class names after the rename lands in `qiskit-ibm-catalog`.
- **Wait for the end of a job.** [`wait_for_final_state()`](#wait-for-the-job-to-end) comes from `qiskit-serverless` PR #2604. Publish that section after a release that includes it.
</Admonition>

A managed workflow job works like a batch job that you submit to an HPC scheduler. `run()` submits the job and returns a `Job` object right away. Use the `Job` object for the tasks that `squeue`, the job's output file, and `scancel` cover on a cluster. The job keeps running after you close your notebook.

The examples on this page use a small workflow that reads one angle, `theta`, and fails if the angle is out of range. Upload it as described in [Deploy your first managed workflow](/docs/managed-workflows/deploy-first-managed-workflow).

In [1]:
# This cell is hidden from users. It creates the working directory.
from pathlib import Path

Path("./my_workflow").mkdir(exist_ok=True)

In [2]:
%%writefile ./my_workflow/my_workflow.py
import math

from qiskit_serverless import get_arguments, save_result

arguments = get_arguments()
theta = arguments["theta"]
if not 0 <= theta <= math.pi:
    raise ValueError(f"theta must be between 0 and pi, got {theta}")

print(f"Computing the expectation value for theta={theta}", flush=True)
save_result({"theta": theta, "expectation_value": math.cos(theta)})

Writing ./my_workflow/my_workflow.py


## Start a job

Get the workflow by its title, and call `run()` with the arguments that the workflow reads:

{/* REVIEW: class rename pending */}
{/* REVIEW: runner value. Keep in sync with the Deploy your first managed workflow page. */}

In [3]:
# This cell is hidden from users. It uploads the workflow under a title that
# other test notebooks do not use, and makes load("my-workflow") return it.
# It also silences the account warning that job.cancel() logs.
import logging

from qiskit_ibm_catalog import (
    QiskitFunction as ManagedWorkflow,
    QiskitServerless as ManagedWorkflowClient,
)

logging.getLogger("qiskit_ibm_runtime").setLevel(logging.ERROR)

_test_title = "docs-run-managed-workflow"
ManagedWorkflowClient().upload(
    ManagedWorkflow(
        title=_test_title,
        entrypoint="my_workflow.py",
        working_dir="./my_workflow/",
        runner="fleets",
    )
)
_load = ManagedWorkflowClient.load
ManagedWorkflowClient.load = lambda self, title, *a, **k: _load(
    self, _test_title if title == "my-workflow" else title, *a, **k
)

<CodeTitle title="run_workflow_remotely.ipynb" />

In [4]:
from qiskit_ibm_catalog import QiskitServerless as ManagedWorkflowClient

client = ManagedWorkflowClient()
my_workflow = client.load("my-workflow")

job = my_workflow.run(theta=0.5)
job_id = job.job_id

`run()` returns after Functions queues the job. Save the job ID so that you can [find the job from a new session](#find-a-job-from-a-new-session).

- To choose which arguments a workflow accepts and check them before the job starts, see [Define inputs and outputs](/docs/managed-workflows/managed-workflow-inputs-and-outputs).
- To run the job on a different compute size, pass `function_size`. See [Choose a size for each run](/docs/managed-workflows/managed-workflow-compute-profiles#choose-a-size-for-each-run).

## Follow a job

### Check the status

Right after you call `run()`, the status is usually `QUEUED`:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [5]:
job.status()

'QUEUED'

| Status | Meaning |
| - | - |
| `QUEUED` | The job waits for a compute slot. |
| `INITIALIZING` | Functions is starting an environment with the job's compute profile. |
| `RUNNING` | The workflow's entrypoint is running. |
| `RUNNING: <stage>` | The entrypoint reported a stage with [`update_status()`](/docs/managed-workflows/manage-quantum-classical-workflows#report-progress-with-update_status), such as `RUNNING: WAITING_FOR_QPU`. |
| `CANCELING` | You canceled the job, and Functions is stopping it. |
| `DONE` | The entrypoint exited successfully. |
| `ERROR` | The entrypoint raised an error or exited with a nonzero code. See [Find the cause of an error](#find-the-cause-of-an-error). |
| `CANCELED` | You canceled the job, or it reached the [24-hour limit](#time-limit). |

`DONE`, `ERROR`, and `CANCELED` are final, and `job.in_terminal_state()` returns `True` for them. A status loop that waits for another value, such as `SUCCEEDED`, never ends.

#### Wait for the job to end

{/* REVIEW: requires a qiskit-serverless release with PR #2604. */}
<Admonition type="note" title="Requires a later release">
`wait_for_final_state()` is waiting on `qiskit-serverless` PR #2604 and is not in `qiskit-serverless` 0.37.0.
</Admonition>

To wait until the job ends and print each status change, call `wait_for_final_state()`. With `timeout` in seconds, it raises a `TimeoutError` if the job is still running, and the job keeps running:

<CodeTitle title="run_workflow_remotely.ipynb" />
```python
job.wait_for_final_state(verbose=True)
```

```text
Waiting for job 3f2a1c0e-6b1d-4c0e-9a51-2f0c4d1e7a90
0:00:00  QUEUED
0:02:10  RUNNING
0:03:05  RUNNING: WAITING_FOR_QPU
0:41:44  DONE
'DONE'
```

### Read the logs

Anything that the workflow writes to standard output or standard error goes to the job's logs. You can read them with `job.logs()` while the job runs, before it has a result:

In [6]:
# This cell is hidden from users. It waits for the job to end, so that the
# logs are complete.
import time

while not job.in_terminal_state():
    time.sleep(5)

<CodeTitle title="run_workflow_remotely.ipynb" />

In [7]:
print(job.logs())

Computing the expectation value for theta=0.5



In [8]:
# This cell is hidden from users. It checks the logs.
assert "Computing the expectation value for theta=0.5" in job.logs()

- Functions uploads new log lines about every 15 seconds, and one last time when the job ends or is canceled. Right after a job starts, `job.logs()` returns `No logs yet.`
- Functions keeps the newest 50 MB of logs for each job, and adds a line at the top that says the logs were truncated.

To decide what your workflow logs, see [Write logs](/docs/managed-workflows/managed-workflow-logs-and-errors#write-logs). To show only the lines that match a regular expression, pass `include` to `job.filtered_logs()`. To hide lines, pass `exclude`.

### Get the result

`job.result()` waits until the job ends, then returns the value that the workflow passed to [`save_result()`](/docs/managed-workflows/managed-workflow-inputs-and-outputs#save-results):

<CodeTitle title="run_workflow_remotely.ipynb" />

In [9]:
job.result()

{'theta': 0.5, 'expectation_value': 0.8775825618903728}

In [10]:
# This cell is hidden from users. It checks the result.
assert _ == {"theta": 0.5, "expectation_value": 0.8775825618903728}, _  # noqa: F821

By default, `job.result()` checks the status every 30 seconds and waits with no time limit. Pass `maxwait` to limit the number of checks. For example, `job.result(maxwait=40)` waits about 20 minutes. To check one time without waiting, pass `wait=False`. If it stops waiting before the job ends, it returns `{}`.

### Check how long each stage took

`job.events()` lists every status and stage change of a job with its time, so you can see how long the job waited for compute and how long each stage took:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [11]:
for event in reversed(job.events()):
    print(
        event.created,
        event.data.get("status") or event.data.get("sub_status"),
    )

2026-11-02T14:43:07.134377Z QUEUED
2026-11-02T14:43:24.474199Z PENDING
2026-11-02T14:46:07.309519Z RUNNING
2026-11-02T14:46:21.115038Z SUCCEEDED


In [12]:
# This cell is hidden from users. It checks the events.
_statuses = [e.data.get("status") for e in job.events()]
assert "QUEUED" in _statuses and "SUCCEEDED" in _statuses, _statuses

The events use stored names: `PENDING` is `INITIALIZING`, and `SUCCEEDED` is `DONE`. For the QPU usage of a run, see [Find the QPU jobs of a run](/docs/managed-workflows/manage-quantum-classical-workflows#find-the-qpu-jobs-of-a-run).

## Find the cause of an error

When the workflow raises an exception, the job ends with the `ERROR` status. For example, run the workflow with an angle that it rejects:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [13]:
failed_job = my_workflow.run(theta=-1.0)

In [14]:
# This cell is hidden from users. It waits for the job to end.
while not failed_job.in_terminal_state():
    time.sleep(10)

<CodeTitle title="run_workflow_remotely.ipynb" />

In [15]:
failed_job.status()

'ERROR'

The logs show the traceback:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [16]:
print(failed_job.logs())

Traceback (most recent call last):
  File "/function_user_data/my_workflow.py", line 8, in <module>
    raise ValueError(f"theta must be between 0 and pi, got {theta}")
ValueError: theta must be between 0 and pi, got -1.0



`job.result()` raises a `QiskitServerlessException`:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [17]:
from qiskit_serverless.exception import QiskitServerlessException

try:
    failed_job.result()
except QiskitServerlessException as error:
    print(error)

    raise ValueError(f"theta must be between 0 and pi, got {theta}")
ValueError: theta must be between 0 and pi, got -1.0



In [18]:
# This cell is hidden from users. It checks the error.
assert failed_job.status() == "ERROR", failed_job.status()
assert (
    "ValueError: theta must be between 0 and pi, got -1.0"
    in failed_job.logs()
)
assert failed_job.events(type="ERROR") == []

If the workflow [reports errors with `send_error()`](/docs/managed-workflows/managed-workflow-logs-and-errors#best-practice-report-errors-with-send_error), read them with `job.events(type="ERROR")`, and read their messages with `job.error_message()`. For a job without reported errors, `job.events(type="ERROR")` returns an empty list, and `job.error_message()` returns `{}`.

{/* REVIEW: for custom workflows on Fleets, a raised ServerlessError is not recorded as an error event: only provider jobs run inside gateway/templates/main.tmpl, which calls send_error(). Custom jobs run the entrypoint directly through fleet_custom_job_wrapper.py. Confirm whether custom jobs should get the same handler, and whether error_message() should return a string instead of {}. */}

<CodeTitle title="run_workflow_remotely.ipynb" />
```python
for event in job.events(type="ERROR"):
    print(event.created, event.data.get("code"), event.data.get("message"))
```

## Manage jobs

### Cancel a job

Cancel a job with `job.cancel()`. The status changes to `CANCELING` while Functions stops the job, and then to `CANCELED`:

<CodeTitle title="run_workflow_remotely.ipynb" />

In [19]:
job_to_cancel = my_workflow.run(theta=1.0)
job_to_cancel.cancel()

'Job has been stopped. No active runtime job ID associated with this serverless job ID.'

In [20]:
# This cell is hidden from users. It checks the cancellation.
while not job_to_cancel.in_terminal_state():
    time.sleep(5)
assert job_to_cancel.status() == "CANCELED", job_to_cancel.status()

`job.cancel()` returns a message about the job and about its [linked QPU jobs](/docs/managed-workflows/manage-quantum-classical-workflows#cancel-linked-qpu-jobs), which it also tries to cancel. For a running job, Functions sends `SIGTERM` to your entrypoint. To save partial results, see [Clean up when a job is canceled](/docs/managed-workflows/managed-workflow-logs-and-errors#clean-up-when-a-job-is-canceled). If the job already ended, `job.cancel()` returns `Job already in terminal state.` and changes nothing.

{/* REVIEW: in a live test (1 Nov 2026), a job canceled while RUNNING went to CANCELING and then DONE, with its result saved, because the entrypoint finished before the stop took effect. Confirm whether a job that finishes during CANCELING should end as DONE or CANCELED. */}

### Find a job from a new session

The `Job` object and the workflow exist only in your current Python session. After you restart a notebook kernel, pass the saved job ID to `client.job()`, or list your recent jobs with `client.jobs()`:

{/* REVIEW: class rename pending */}

<CodeTitle title="run_workflow_remotely.ipynb" />

In [21]:
from qiskit_ibm_catalog import QiskitServerless as ManagedWorkflowClient

client = ManagedWorkflowClient()
job = client.job(job_id)
my_workflow = client.load("my-workflow")

client.jobs(function=my_workflow, status="ERROR", limit=5)

[<Job | e6b1ffa1-cd26-437d-bd53-b265dd4ba952>,
 <Job | 11308f2a-8d7a-40a0-a970-7fdeca2b171b>,
 <Job | af597785-3470-469a-9051-607e0185892e>,
 <Job | a9051931-3bc2-4b80-9ce6-c39d28ce260c>,
 <Job | e329e8e3-e955-4188-9bfd-8f3050ee2d39>]

In [22]:
# This cell is hidden from users. It checks the job lookups.
assert job.result()["theta"] == 0.5
assert failed_job.job_id in [j.job_id for j in _]  # noqa: F821

`client.jobs()` returns your 10 most recent jobs, newest first. Filter it with `function`, `status` (from the [status table](#check-the-status)), or `created_after` (an ISO 8601 timestamp), and page through it with `limit` and `offset`. To list the workflows that you uploaded, call `client.list()`.

The [Workloads](https://quantum.cloud.ibm.com/workloads) table on IBM Quantum Platform also lists your jobs, with their status, compute profile, and run times.

{/* REVIEW: the gateway mirrors Fleets jobs to the Runtime API as workloads (core/domain/workload_payload.py) only when the workloads.mirror.enabled setting allows it (default "false"). Confirm that the setting is on in production for custom workflows, and how the Workloads table shows the QPU jobs of a workflow. */}

### Time limit

Functions cancels a job that runs for more than 24 hours, and the job ends with the `CANCELED` status. During long runs, [save checkpoints to file storage](/docs/managed-workflows/manage-files-across-managed-workflows#best-practice-checkpoint-long-jobs) so that work is not lost.

{/* REVIEW: the timeout is measured from the job's latest event, not from its start (gateway/scheduler/tasks/update_fleets_jobs_statuses.py stop_job_if_timeout). Calls to update_status() and send_error() create events, so a job that reports stages can run longer than 24 hours. This looks like a bug; confirm the intended behavior. */}

## Troubleshoot jobs

| Problem | Cause |
| - | - |
| `run()` fails with `Code: 429` and `Active job limit reached.` | You reached the [active job limit](/docs/managed-workflows/run-parameter-sweep#active-job-limit). Wait for some jobs to end, or [cancel](#cancel-a-job) jobs that you no longer need. |
| `job.logs()` returns `No logs yet.` | The job is queued or just started. Wait about 15 seconds. |
| `job.result()` returns `{}` | The job has not ended, was canceled, or saved no result. See [Save results](/docs/managed-workflows/managed-workflow-inputs-and-outputs#save-results). |
| `job.error_message()` returns `{}` | The workflow raised an exception without reporting an error. Read the traceback with `job.logs()`. |
| The job is `CANCELED`, but you did not cancel it | The job reached the [24-hour limit](#time-limit). |

## Next steps

<Admonition type="tip" title="Recommendations">
- [Write logs and handle errors](/docs/managed-workflows/managed-workflow-logs-and-errors) to set up the logs and errors that a job reports.
- [Run a parameter sweep](/docs/managed-workflows/run-parameter-sweep) to run the workflow one time for each set of inputs.
</Admonition>